# 📚 RAG (Retrieval-Augmented Generation) — Hands-On Lab## Build a working RAG system from scratch. Test every component.**What this notebook does:**- Build a COMPLETE RAG pipeline in ~20 lines- Test 4 chunking strategies side by side- Compare dense vs sparse vs hybrid search- Implement reranking and see the accuracy boost- Advanced: multi-query, CRAG (corrective RAG)- Evaluate with RAGAS metrics- Multimodal RAG: tables + images as text**Setup:**```pip install openai chromadb langchain langchain-openai rank_bm25 numpy```**Cost:** ~$0.30 - $0.50 to run everything

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# pip install openai chromadb langchain langchain-openai rank_bm25 numpy
from openai import OpenAI
import numpy as np
import time, json, hashlib

client = OpenAI()

def ask(prompt, model="gpt-4o-mini", temperature=0, system=None):
    messages = []
    if system: messages.append({"role":"system","content":system})
    messages.append({"role":"user","content":prompt})
    r = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    return r.choices[0].message.content.strip(), r.usage.total_tokens

def embed(texts):
    """Turn a list of texts into embeddings (lists of numbers)."""
    if isinstance(texts, str): texts = [texts]
    r = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [np.array(d.embedding) for d in r.data]

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Quick test
answer, tokens = ask("Say 'RAG is ready' in 3 words.")
print(f"✅ Setup complete! {answer} (used {tokens} tokens)")

---# 1️⃣ Complete RAG Pipeline (From Scratch)**Analogy:** An open-book exam. You don't memorize everything — you FIND the right page and read the answer.**What we build:** Documents → Chunk → Embed → Store → Search → Generate answer. The whole thing in ~30 lines.

In [ ]:
# ============================================================
# FULL RAG PIPELINE: 7 steps in one function
# Think: open-book exam. Find the page, read the answer.
# ============================================================

# ---------- STEP 1: Our "knowledge base" (your company docs) ----------
documents = [
    "REFUND POLICY: Standard items can be returned within 30 days of purchase. "
    "Electronics have a shorter return window of 15 days. All items must be in "
    "original unused condition with receipt. Opened software is non-refundable.",
    
    "PRICING: Pro plan costs $49 per month. Enterprise plan costs $199 per month. "
    "Annual billing gives 20% discount. Student discount is 50% off Pro plan. "
    "Non-profit organizations get 30% off any plan.",
    
    "WARRANTY: All products come with 1-year manufacturer warranty covering defects. "
    "Physical damage is NOT covered by warranty. Extended warranty available for "
    "$99/year. Warranty claims require proof of purchase.",
    
    "SHIPPING: Free shipping on orders over $50. Standard shipping takes 5-7 business "
    "days. Express shipping (2-day) costs $15. International shipping available to "
    "30 countries. Tracking number provided within 24 hours.",
]

# ---------- STEP 2 & 3: Chunk + Embed ----------
# (These docs are already small enough to be chunks)
print("Step 1-3: Loading, chunking, and embedding documents...")
doc_embeddings = embed(documents)
print(f"  ✅ {len(documents)} documents embedded ({len(doc_embeddings[0])} dimensions each)")

# ---------- STEP 4: Store (simple in-memory for demo) ----------
# In production: use ChromaDB, Qdrant, or Pinecone
knowledge_base = list(zip(documents, doc_embeddings))

# ---------- STEP 5: Search (find the most relevant docs) ----------
def search(query, top_k=2):
    """Find the most relevant documents for a question."""
    query_emb = embed(query)[0]
    
    # Score every document by similarity to the query
    scored = []
    for doc, doc_emb in knowledge_base:
        score = cosine(query_emb, doc_emb)
        scored.append((score, doc))
    
    # Sort by score (highest first) and return top K
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

# ---------- STEP 6 & 7: Augment + Generate ----------
def rag_answer(question):
    """The complete RAG pipeline: search docs, then answer using them."""
    
    # Search for relevant documents
    results = search(question, top_k=2)
    
    # Build the context from retrieved docs
    context = "\n\n".join([doc for score, doc in results])
    
    # Ask the LLM to answer ONLY from the context
    prompt = f"""Answer the question using ONLY the context below.
If the answer is NOT in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""
    
    answer, tokens = ask(prompt)
    return answer, results, tokens


# ---------- TEST IT ----------
print("\n" + "=" * 65)
print("COMPLETE RAG PIPELINE")
print("=" * 65)

test_questions = [
    "Can I return electronics after 20 days?",
    "How much is the Pro plan with annual billing?",
    "Is physical damage covered by warranty?",
    "What is your shipping policy to Canada?",      # Answerable
    "What programming language do you use?",         # NOT in docs!
]

for q in test_questions:
    answer, results, tokens = rag_answer(q)
    top_score = results[0][0]
    
    print(f"\n  Q: {q}")
    print(f"  Best doc match: {top_score:.3f} similarity")
    print(f"  A: {answer}")

print("\n💡 The LLM answers from YOUR documents, not from its training data.")
print("💡 Question not in docs? It says 'I don't have that information.' (no hallucination)")

---# 2️⃣ Chunking Strategies (How You Cut = How Good Your Answers)**Analogy:** Making flash cards from a textbook. If you cut a page in the middle of a sentence — "The refund policy is" on one card and "30 days" on another — neither card is useful alone.**What we test:** 4 strategies on the SAME document. See how each one splits differently.

In [ ]:
# ============================================================
# CHUNKING: 4 ways to split documents into pieces
# The quality of your chunks = the quality of your answers
# ============================================================

# A realistic document to split
long_document = """ACME CORPORATION REFUND AND RETURN POLICY

Section 1: Standard Returns
All standard items purchased from Acme Corporation may be returned within 30 days of the original purchase date. Items must be in their original, unused condition with all tags attached. A valid receipt or proof of purchase is required for all returns.

Section 2: Electronics Returns  
Electronic items including laptops, tablets, phones, and accessories have a reduced return window of 15 days from purchase. Electronics must be returned in original packaging with all included accessories. Opened software and digital downloads are non-refundable under any circumstances.

Section 3: Refund Processing
Refunds are processed within 5-7 business days after the returned item is received and inspected. Credit card refunds appear on your statement within 1-2 billing cycles. Cash purchases over $50 are refunded by company check mailed within 10 business days.

Section 4: Exceptions
Holiday purchases made between November 15 and December 31 may be returned until January 31 of the following year. Clearance items marked as final sale cannot be returned. Gift cards and prepaid cards are non-refundable."""

print(f"Document: {len(long_document)} characters, {len(long_document.split())} words\n")


# ---------- METHOD 1: Fixed-size (the dumb way) ----------
def chunk_fixed(text, size=200):
    """Cut every N characters. Fast but might split mid-sentence."""
    chunks = []
    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])
    return chunks

# ---------- METHOD 2: Recursive (the smart default) ----------
def chunk_recursive(text, max_size=300, overlap=50):
    """Split at paragraph breaks first, then sentences, then words."""
    # First try splitting by paragraphs
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) > max_size and current:
            chunks.append(current.strip())
            # Keep overlap from the end of previous chunk
            words = current.split()
            overlap_text = " ".join(words[-10:]) if len(words) > 10 else ""
            current = overlap_text + " " + para
        else:
            current = current + "\n\n" + para if current else para
    if current:
        chunks.append(current.strip())
    return chunks

# ---------- METHOD 3: Sentence-based ----------
def chunk_by_sentence(text, sentences_per_chunk=3):
    """Group complete sentences together."""
    # Simple sentence splitter
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " "))
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i+sentences_per_chunk])
        chunks.append(chunk)
    return chunks


# ---------- COMPARE ALL 3 ----------

print("CHUNKING COMPARISON")
print("=" * 65)

for name, func in [
    ("Fixed (200 chars)", lambda: chunk_fixed(long_document, 200)),
    ("Recursive (300 chars)", lambda: chunk_recursive(long_document, 300)),
    ("Sentence (3 per chunk)", lambda: chunk_by_sentence(long_document, 3)),
]:
    chunks = func()
    print(f"\n{name}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        # Show first 80 chars of each chunk
        preview = chunk.replace("\n", " ")[:80]
        print(f"  Chunk {i+1} ({len(chunk):3d} chars): {preview}...")

print("\n💡 Fixed: cuts mid-sentence ('Electronics must be retur...'). Bad.")
print("💡 Recursive: respects paragraph boundaries. Each chunk is a complete idea. Good.")
print("💡 Sentence: complete sentences but might split related info. Medium.")
print("💡 USE RECURSIVE for 90% of cases.")

---# 3️⃣ Embeddings — Turn Words into Numbers**Analogy:** Imagine organizing a library by MEANING, not alphabetically. "Machine learning" and "AI algorithms" are shelved next to each other because they mean similar things. "Chocolate cake recipe" is on a completely different shelf.**What we test:** Embed 6 texts. See which ones the model considers similar vs different.

In [ ]:
# ============================================================
# EMBEDDINGS: Convert text to numbers. Similar meaning = similar numbers.
# ============================================================

texts = [
    "Machine learning uses data to find patterns",      # Tech/AI
    "AI algorithms learn from training datasets",        # Tech/AI (similar!)
    "Deep neural networks process information",          # Tech/AI (similar!)
    "The weather forecast predicts rain tomorrow",       # Weather (different!)
    "How to bake a chocolate cake at home",              # Cooking (different!)
    "Artificial intelligence is transforming industry",  # Tech/AI (similar!)
]

print("EMBEDDING SIMILARITY MATRIX")
print("=" * 65)
print("Each text is converted to 1536 numbers. Similar meaning = similar numbers.\n")

# Embed all texts
embeddings = embed(texts)

# Compare every pair
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i+1}]", end="")
print()

for i, (text_a, emb_a) in enumerate(zip(texts, embeddings)):
    label = text_a[:42] + "..."
    print(f"  [{i+1}] {label:>42}", end="")
    for j, emb_b in enumerate(embeddings):
        sim = cosine(emb_a, emb_b)
        # Color-code: high similarity = highlight
        marker = "██" if sim > 0.8 and i != j else f"{sim:.2f}"[1:]  # Remove leading 0
        print(f" {marker:>5}", end="")
    print()

print()
print("  ██ = HIGH similarity (>0.80) — model knows these are related!")
print()
print("  💡 'Machine learning' and 'AI algorithms' are HIGH — same topic, different words.")
print("  💡 'Machine learning' and 'chocolate cake' are LOW — completely different topics.")
print("  💡 This is HOW semantic search works: match by meaning, not keywords.")

---# 4️⃣ Dense vs Sparse vs Hybrid Search**Analogy:** Two librarians. One understands MEANING ("automobile" matches "car"). The other matches exact WORDS ("error 404" matches "404 error"). Using BOTH = best results.**What we test:** Same query, 3 search methods. See what each finds and misses.

In [ ]:
# ============================================================
# SEARCH COMPARISON: Dense (meaning) vs Sparse (keywords) vs Hybrid
# ============================================================

from collections import Counter
import math

# Knowledge base
docs = [
    "Python error code 404 means the requested page was not found on the server",
    "Machine learning algorithms learn patterns from training data automatically",
    "HTTP 404 status code indicates the requested resource is missing",
    "Artificial intelligence is transforming how businesses operate globally",
    "Debug Python errors by checking the full stack trace and error messages",
    "Neural networks are a subset of machine learning inspired by the brain",
]

# ---------- DENSE SEARCH (meaning-based, uses embeddings) ----------
doc_embeds = embed(docs)

def dense_search(query, top_k=3):
    q_emb = embed(query)[0]
    scored = [(cosine(q_emb, d_emb), doc) for doc, d_emb in zip(docs, doc_embeds)]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

# ---------- SPARSE SEARCH / BM25 (keyword-based) ----------
def simple_bm25(query, documents, k1=1.5, b=0.75):
    """Simplified BM25: scores documents by keyword overlap."""
    query_terms = query.lower().split()
    doc_tokens = [d.lower().split() for d in documents]
    avg_dl = sum(len(d) for d in doc_tokens) / len(doc_tokens)
    N = len(documents)
    
    scores = []
    for doc_idx, doc_toks in enumerate(doc_tokens):
        score = 0
        dl = len(doc_toks)
        tf_map = Counter(doc_toks)
        for term in query_terms:
            tf = tf_map.get(term, 0)
            df = sum(1 for d in doc_tokens if term in d)
            if df == 0: continue
            idf = math.log((N - df + 0.5) / (df + 0.5) + 1)
            tf_score = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avg_dl))
            score += idf * tf_score
        scores.append((score, documents[doc_idx]))
    
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:3]

# ---------- HYBRID (combine both) ----------
def hybrid_search(query, top_k=3):
    dense = dense_search(query, top_k=5)
    sparse = simple_bm25(query, docs)
    
    # RRF (Reciprocal Rank Fusion) to merge
    rrf_scores = {}
    for rank, (score, doc) in enumerate(dense):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (rank + 60)
    for rank, (score, doc) in enumerate(sparse):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (rank + 60)
    
    combined = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return combined[:top_k]


# ---------- COMPARE ----------
query = "Python 404 error"

print("SEARCH COMPARISON")
print("=" * 65)
print(f"Query: '{query}'\n")

print("DENSE (meaning-based):")
for score, doc in dense_search(query):
    match = "✅" if "404" in doc else "  "
    print(f"  {match} [{score:.3f}] {doc[:65]}...")

print("\nSPARSE / BM25 (keyword-based):")
for score, doc in simple_bm25(query, docs):
    match = "✅" if "404" in doc else "  "
    print(f"  {match} [{score:.3f}] {doc[:65]}...")

print("\nHYBRID (dense + sparse combined):")
for doc, score in hybrid_search(query):
    match = "✅" if "404" in doc else "  "
    print(f"  {match} [{score:.4f}] {doc[:65]}...")

print("\n💡 Dense: understands 'Python error' = 'debug Python' but may miss exact '404'.")
print("💡 Sparse: nails exact keyword '404' but misses synonyms.")
print("💡 Hybrid: catches BOTH. This is what production systems use.")

---# 5️⃣ Advanced RAG — Multi-Query + CRAG**Multi-Query:** Same question rephrased 3 ways = find more documents. Like searching Google with 3 different queries.**CRAG (Corrective RAG):** After retrieving docs, GRADE each one: "Is this actually relevant?" Throw away the irrelevant ones before the LLM sees them.

In [ ]:
# ============================================================
# MULTI-QUERY: Rephrase question 3 ways, search each, combine results
# CRAG: Grade retrieved docs, throw away irrelevant ones
# ============================================================

# Use the same docs and search from above

# ---------- MULTI-QUERY ----------
def multi_query_search(question, top_k=3):
    """
    Step 1: LLM rephrases the question 3 different ways
    Step 2: Search for EACH version
    Step 3: Combine all results (deduplicated)
    """
    
    # Step 1: Generate variations
    variations_text, _ = ask(
        f"Generate 3 different search queries for: '{question}'\nOne per line. No numbering.",
        temperature=0.7
    )
    variations = [v.strip() for v in variations_text.strip().split("\n") if v.strip()][:3]
    
    print(f"  Original: {question}")
    print(f"  Variations:")
    for v in variations:
        print(f"    → {v}")
    
    # Step 2: Search for each variation
    all_docs = set()
    for q in [question] + variations:
        for score, doc in dense_search(q, top_k=2):
            all_docs.add(doc[:80])  # Dedup by first 80 chars
    
    print(f"  Single search: 2 docs | Multi-query: {len(all_docs)} docs")
    return list(all_docs)


# ---------- CRAG (Corrective RAG) ----------
def crag_filter(question, retrieved_docs):
    """
    Grade each retrieved document: is it RELEVANT to the question?
    Throw away irrelevant ones BEFORE the LLM tries to answer.
    """
    
    kept = []
    tossed = []
    
    for doc in retrieved_docs:
        grade, _ = ask(
            f"Is this document relevant to the question: '{question}'?\n"
            f"Document: {doc}\n"
            f"Answer YES or NO only."
        )
        
        if "YES" in grade.upper():
            kept.append(doc)
        else:
            tossed.append(doc)
    
    return kept, tossed


# ---------- TEST MULTI-QUERY ----------
print("MULTI-QUERY RETRIEVAL")
print("=" * 65)
print()
results = multi_query_search("How to fix credential recovery issues")

# ---------- TEST CRAG ----------
print("\n\nCRAG (CORRECTIVE RAG)")
print("=" * 65)
print()

question = "What is the refund policy for electronics?"
# Simulate retrieval (some relevant, some not)
retrieved = [
    "Electronics have a 15-day return window. Must be in original packaging.",
    "Standard items can be returned within 30 days with receipt.",
    "Free shipping on orders over $50. Express costs $15.",  # NOT relevant!
]

print(f"  Question: {question}")
print(f"  Retrieved {len(retrieved)} docs. Grading each...\n")

kept, tossed = crag_filter(question, retrieved)

for doc in kept:
    print(f"  ✅ KEEP: {doc[:65]}...")
for doc in tossed:
    print(f"  ❌ TOSS: {doc[:65]}...")

print(f"\n  Before CRAG: {len(retrieved)} docs (including irrelevant noise)")
print(f"  After CRAG:  {len(kept)} docs (only relevant ones)")
print(f"\n💡 Multi-query: finds 30-50% more relevant docs by rephrasing.")
print(f"💡 CRAG: removes irrelevant docs that would confuse the LLM.")
print(f"💡 Together: more relevant docs + less noise = better answers.")

---# 6️⃣ Multimodal RAG — Tables + Images as Text**Analogy:** A regular library only catalogs books (text). A multimedia library also catalogs paintings (images) and data tables. You want the universal one.**What we do:** Convert a table to markdown + describe an image as text. Both become searchable in the same vector store.

In [ ]:
# ============================================================
# MULTIMODAL RAG: Tables and images become searchable text
# Strategy: convert EVERYTHING to text, embed it all the same way
# ============================================================

# ---------- TABLE → MARKDOWN ----------
print("TABLE → MARKDOWN")
print("=" * 65)

table_data = [
    ["Region", "Q3 Revenue", "Growth", "Top Product"],
    ["North America", "$45M", "+12%", "Widget Pro"],
    ["Europe", "$32M", "+8%", "Widget Lite"],
    ["Asia Pacific", "$28M", "+22%", "Widget Enterprise"],
    ["Latin America", "$12M", "+35%", "Widget Starter"],
]

# Convert to markdown format (LLMs understand this perfectly)
md_table = "| " + " | ".join(table_data[0]) + " |\n"
md_table += "|" + "|".join(["---"] * len(table_data[0])) + "|\n"
for row in table_data[1:]:
    md_table += "| " + " | ".join(row) + " |\n"

print("Original table → Markdown:\n")
print(md_table)

# Now this markdown is a text chunk, embeddable and searchable!
table_chunk = f"Q3 Regional Revenue Data:\n{md_table}"

# ---------- IMAGE → TEXT DESCRIPTION ----------
print("\nIMAGE → TEXT DESCRIPTION")
print("=" * 65)

# In production: send actual image to GPT-4o
# image_desc = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[{"role":"user","content":[
#         {"type":"text","text":"Describe this chart for a search system."},
#         {"type":"image_url","image_url":{"url":"data:image/png;base64,..."}}
#     ]}]
# )

# Simulated description (what GPT-4o would say about a chart)
image_description = (
    "Bar chart showing Q3 2024 regional revenue. North America leads at $45M "
    "(+12% YoY), followed by Europe at $32M (+8%), Asia Pacific at $28M (+22%), "
    "and Latin America at $12M (+35%). Asia Pacific shows fastest growth rate."
)
print(f"  GPT-4o would describe the chart as:\n  '{image_description[:100]}...'\n")

# ---------- SEARCH ACROSS ALL CONTENT TYPES ----------
print("UNIFIED SEARCH (text + table + image description)")
print("=" * 65)

# All content types become text chunks
all_chunks = [
    "Standard items can be returned within 30 days of purchase.",     # Regular text
    table_chunk,                                                       # Table as markdown  
    f"Chart description: {image_description}",                        # Image as text
    "Pro plan costs $49/month. Enterprise costs $199/month.",         # Regular text
]

chunk_embeds = embed(all_chunks)

# Search!
query = "Which region grew fastest in Q3?"
q_emb = embed(query)[0]

print(f"\n  Query: '{query}'\n")
scored = [(cosine(q_emb, ce), chunk[:80]) for chunk, ce in zip(all_chunks, chunk_embeds)]
scored.sort(key=lambda x: x[0], reverse=True)

for score, chunk in scored:
    source = "📊 TABLE" if "Region" in chunk else "📈 CHART" if "Chart" in chunk else "📝 TEXT"
    print(f"  [{score:.3f}] {source}: {chunk}...")

print("\n💡 The query found BOTH the table AND chart description — different formats, same topic.")
print("💡 Strategy: convert everything to text. 90% of the value, zero infrastructure changes.")

---# 7️⃣ RAG Evaluation (Build Your Own RAGAS)**Analogy:** 4 things to check in an open-book exam:1. Did the student open the RIGHT pages? (Precision)2. Did they find ALL relevant pages? (Recall)3. Did they answer from the book, not memory? (Faithfulness)4. Did they actually answer the question? (Relevance)**What we build:** Our own evaluation system. No external library needed.

In [ ]:
# ============================================================
# RAG EVALUATION: 4 metrics to check if your RAG is working
# We build the evaluation from scratch so you understand each metric
# ============================================================

def evaluate_rag(question, expected_answer, retrieved_docs, generated_answer):
    """
    Score a single RAG result on 4 dimensions.
    Returns a dict with scores and explanations.
    """
    
    results = {}
    
    # 1. CONTEXT PRECISION: Are the retrieved docs relevant?
    #    "Did the librarian find the RIGHT books?"
    relevant_count = 0
    for doc in retrieved_docs:
        grade, _ = ask(
            f"Is this document relevant to: '{question}'?\n"
            f"Document: {doc}\nAnswer YES or NO only."
        )
        if "YES" in grade.upper():
            relevant_count += 1
    
    results["precision"] = relevant_count / max(len(retrieved_docs), 1)
    
    # 2. FAITHFULNESS: Does the answer use ONLY the provided docs?
    #    "Did the student answer from the book, not from memory?"
    faith_check, _ = ask(
        f"Is this answer fully supported by the context? No made-up facts?\n"
        f"Context: {' '.join(retrieved_docs)}\n"
        f"Answer: {generated_answer}\n"
        f"Score 1-10 where 10 = fully faithful, 1 = completely made up. Number only."
    )
    try:
        results["faithfulness"] = int(faith_check.strip()) / 10
    except:
        results["faithfulness"] = 0.5
    
    # 3. ANSWER RELEVANCE: Does the answer address the question?
    #    "Did the student actually answer the question?"
    relevance_check, _ = ask(
        f"Does this answer address the question?\n"
        f"Question: {question}\n"
        f"Answer: {generated_answer}\n"
        f"Score 1-10 where 10 = perfectly answers, 1 = completely off topic. Number only."
    )
    try:
        results["relevance"] = int(relevance_check.strip()) / 10
    except:
        results["relevance"] = 0.5
    
    # 4. CORRECTNESS: Does it match the expected answer?
    correct_check, _ = ask(
        f"Do these two answers convey the same information?\n"
        f"Expected: {expected_answer}\n"
        f"Generated: {generated_answer}\n"
        f"Score 1-10 where 10 = identical meaning, 1 = contradictory. Number only."
    )
    try:
        results["correctness"] = int(correct_check.strip()) / 10
    except:
        results["correctness"] = 0.5
    
    return results


# ---------- TEST ----------

eval_cases = [
    {
        "question": "Can I return electronics after 20 days?",
        "expected": "No, electronics must be returned within 15 days.",
        "retrieved": [
            "Electronics have a 15-day return window. Must be in original packaging.",
            "Standard items can be returned within 30 days."
        ],
        "generated": "No, electronics have a shorter return window of 15 days, so a return after 20 days would not be accepted."
    },
    {
        "question": "How much is Pro plan annually?",
        "expected": "$49/month with 20% annual discount = $470.40/year",
        "retrieved": [
            "Pro plan costs $49/month. Annual billing gives 20% discount.",
            "Free shipping on orders over $50."  # Irrelevant doc!
        ],
        "generated": "The Pro plan costs $49/month. With the 20% annual discount, it would be $470.40 per year."
    },
]

print("RAG EVALUATION REPORT")
print("=" * 65)

for case in eval_cases:
    print(f"\n  Q: {case['question']}")
    scores = evaluate_rag(
        case["question"], case["expected"],
        case["retrieved"], case["generated"]
    )
    
    for metric, score in scores.items():
        grade = "🟢" if score >= 0.8 else "🟡" if score >= 0.6 else "🔴"
        bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
        print(f"    {grade} {metric:15s} {bar} {score:.0%}")

print("\n\nDIAGNOSTIC MAP (memorize this!):")
print("  🔴 Low precision  → search is sloppy (fix chunking, add reranking)")
print("  🔴 Low recall      → search is missing docs (add hybrid, multi-query)")
print("  🔴 Low faithfulness → LLM is making stuff up (fix prompt, add citations)")
print("  🔴 Low relevance   → LLM is off topic (fix prompt, better model)")
print("\n💡 Faithfulness < 90% means your system is LYING. Fix it immediately.")
print("💡 Run this evaluation in CI/CD. Block deploys if metrics drop.")

---# 🏁 Summary — RAG Architecture Decision Guide| Component | Start With | Upgrade To | When to Upgrade ||-----------|-----------|------------|-----------------|| **Chunking** | Recursive (500 words) | Parent-Child | Need precise search + rich context || **Embeddings** | OpenAI text-embedding-3-small | Jina v3, BGE-large | Need local/free, long text || **Vector DB** | ChromaDB (in-memory) | Qdrant, pgvector | Going to production || **Search** | Dense only | Hybrid (dense + BM25) | Missing keyword matches || **Reranking** | None | Cohere, cross-encoder | Precision below 80% || **Advanced** | None | Multi-query, CRAG | Low recall, noisy results || **Evaluation** | Manual spot-check | RAGAS in CI/CD | Going to production |**The golden rule:** Fix chunking first. Then search. Then generation. In that order.